# 01 — Exploratory Data Analysis

**Dataset:** UCI Online Retail (541,909 transactions, Dec 2010 – Dec 2011)  
**Tools:** DuckDB (SQL aggregations), Polars (transformations), Plotly (charts)  
**Goal:** Understand the dataset shape, quality, distributions, and anomalies before building features.

---

### Sections
1. Load & inspect raw data
2. Data quality analysis (nulls, returns, anomalies)
3. Revenue & transaction distributions
4. Customer behaviour overview
5. Geographic analysis
6. Product category analysis
7. Temporal patterns
8. Customer segmentation preview (RFM quartiles)


In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────
import sys
from pathlib import Path

import duckdb
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import polars as pl
from IPython.display import display
from rich import print as rprint

# Add project root to path
sys.path.insert(0, str(Path().resolve().parent))

from backend.config import settings
from backend.data.load_data import load_uci_csv
from backend.features.rfm import clean_transactions, assign_product_categories, assign_amount_buckets
from backend.features.duckdb_agg import DuckDBAggregator

# Plotly template
import plotly.io as pio
pio.templates.default = 'plotly_white'

print('✓ Imports OK')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────
CSV_PATH = settings.UCI_CSV_PATH
OBS_END   = '2011-06-30'   # end of 6-month observation window
HOLD_END  = '2011-12-09'   # end of holdout window

print(f'CSV path:  {CSV_PATH}')
print(f'Obs end:   {OBS_END}')
print(f'Hold end:  {HOLD_END}')

## 1. Load & Inspect Raw Data

In [ ]:
# Load raw data via Polars
raw_df = load_uci_csv(CSV_PATH)

print(f'Shape:   {raw_df.shape}')
print(f'Columns: {raw_df.columns}')
print(f'Memory:  {raw_df.estimated_size("mb"):.1f} MB')
raw_df.head(5)

In [ ]:
# Schema and null summary
schema_report = pl.DataFrame({
    'column':    raw_df.columns,
    'dtype':     [str(d) for d in raw_df.dtypes],
    'null_count':[raw_df[c].null_count() for c in raw_df.columns],
    'null_pct':  [round(100 * raw_df[c].null_count() / len(raw_df), 2) for c in raw_df.columns],
    'unique':    [raw_df[c].n_unique() for c in raw_df.columns],
})
display(schema_report)

In [ ]:
# DuckDB basic stats
with DuckDBAggregator() as duck:
    duck.register_polars('raw', raw_df)
    stats = duck.agg_basic_stats('raw')
    
display(stats)

## 2. Data Quality Analysis

In [ ]:
# Identify data issues
issues = {
    'null_customer_id':      raw_df['customer_id'].null_count(),
    'cancellations_C':       raw_df.filter(pl.col('invoice_no').str.starts_with('C')).height,
    'zero_or_neg_quantity':  raw_df.filter(pl.col('quantity') <= 0).height,
    'zero_or_neg_price':     raw_df.filter(pl.col('unit_price') <= 0).height,
    'extreme_price':         raw_df.filter(pl.col('unit_price') > 10_000).height,
    'duplicate_invoice_rows':raw_df.filter(
        pl.struct(['invoice_no','stock_code']).is_duplicated()
    ).height,
}

for k, v in issues.items():
    pct = 100 * v / len(raw_df)
    rprint(f'  {k:<30} {v:>7,}  ({pct:.2f}%)')

In [ ]:
# Distribution of unit prices (log scale)
prices = raw_df.filter(
    pl.col('unit_price') > 0
)['unit_price'].to_list()

fig = px.histogram(
    x=prices, nbins=80,
    title='Unit Price Distribution (log scale)',
    labels={'x': 'Unit Price (£)'},
    log_y=True,
)
fig.show()

In [ ]:
# Quantity distribution
qty = raw_df['quantity'].to_list()
fig = px.histogram(
    x=qty, nbins=60,
    title='Quantity Distribution',
    labels={'x': 'Quantity'},
    range_x=[-100, 500],
)
fig.show()

In [ ]:
# Apply cleaning and compare
cleaned = clean_transactions(raw_df)
cleaned = assign_product_categories(cleaned)
cleaned = assign_amount_buckets(cleaned)

print(f'Raw rows:     {len(raw_df):>8,}')
print(f'Cleaned rows: {len(cleaned):>8,}  ({100*len(cleaned)/len(raw_df):.1f}% retained)')
print(f'\nCleaned stats:')
print(f'  Unique customers: {cleaned["customer_id"].n_unique():,}')
print(f'  Unique invoices:  {cleaned["invoice_no"].n_unique():,}')
print(f'  Unique products:  {cleaned["stock_code"].n_unique():,}')
print(f'  Total revenue:    £{(cleaned["quantity"] * cleaned["unit_price"]).sum():,.2f}')

## 3. Revenue & Transaction Distributions

In [ ]:
# Monthly revenue
with DuckDBAggregator() as duck:
    duck.register_polars('transactions', cleaned)
    monthly = duck.agg_monthly_revenue('transactions')

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=['Monthly Revenue (£)', 'Monthly Active Customers'],
)
months = monthly['month'].to_list()
fig.add_trace(go.Bar(x=months, y=monthly['revenue'].to_list(), name='Revenue'), row=1, col=1)
fig.add_trace(go.Scatter(x=months, y=monthly['active_customers'].to_list(),
                          mode='lines+markers', name='Customers'), row=2, col=1)
fig.update_layout(height=500, showlegend=False, title='Monthly Revenue & Activity')
fig.show()

In [ ]:
# Order value distribution (capped at 99th percentile)
with DuckDBAggregator() as duck:
    duck.register_polars('transactions', cleaned)
    order_values = duck.query("""
        SELECT
            invoice_no,
            customer_id,
            SUM(quantity * unit_price) AS order_value
        FROM transactions
        GROUP BY invoice_no, customer_id
        HAVING SUM(quantity * unit_price) > 0
    """)

p99 = order_values['order_value'].quantile(0.99)
trimmed = order_values.filter(pl.col('order_value') <= p99)

fig = px.histogram(
    trimmed.to_pandas(), x='order_value', nbins=80,
    title=f'Order Value Distribution (≤ £{p99:.0f} = 99th pct)',
    labels={'order_value': 'Order Value (£)'},
)
fig.show()

print(f'Mean order value:   £{order_values["order_value"].mean():.2f}')
print(f'Median order value: £{order_values["order_value"].median():.2f}')
print(f'P90 order value:    £{order_values["order_value"].quantile(0.90):.2f}')
print(f'P99 order value:    £{p99:.2f}')

## 4. Customer Behaviour Overview

In [ ]:
# Customer-level totals
with DuckDBAggregator() as duck:
    duck.register_polars('transactions', cleaned)
    cust_totals = duck.agg_customer_totals('transactions')

print(f'Total customers: {len(cust_totals):,}')
print(f'\nOrder frequency distribution:')
print(cust_totals['total_orders'].describe())

In [ ]:
# Purchase frequency distribution
fig = px.histogram(
    cust_totals.filter(pl.col('total_orders') <= 40).to_pandas(),
    x='total_orders', nbins=40,
    title='Customer Purchase Frequency Distribution (≤40 orders)',
    labels={'total_orders': 'Total Orders'},
)
fig.show()

one_time = (cust_totals['total_orders'] == 1).sum()
print(f'\nOne-time buyers:   {one_time:,} ({100*one_time/len(cust_totals):.1f}%)')
print(f'Repeat buyers:     {len(cust_totals)-one_time:,}')

In [ ]:
# Revenue concentration — Lorenz curve
rev_sorted = cust_totals['total_revenue'].sort(descending=False).to_numpy()
cum_rev = np.cumsum(rev_sorted) / rev_sorted.sum()
cum_cust = np.arange(1, len(rev_sorted)+1) / len(rev_sorted)

# Pareto: top X% of customers = Y% of revenue
top20_idx = int(0.80 * len(rev_sorted))
top20_rev_pct = (1 - cum_rev[top20_idx]) * 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=cum_cust, y=cum_rev, mode='lines', name='Actual',
                          line=dict(color='steelblue', width=2)))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Perfect Equality',
                          line=dict(color='grey', dash='dash')))
fig.update_layout(
    title=f'Lorenz Curve — Top 20% of customers → {top20_rev_pct:.1f}% of revenue',
    xaxis_title='Cumulative % of Customers',
    yaxis_title='Cumulative % of Revenue',
)
fig.show()

## 5. Geographic Analysis

In [ ]:
with DuckDBAggregator() as duck:
    duck.register_polars('transactions', cleaned)
    country_rev = duck.agg_country_revenue('transactions')

fig = px.bar(
    country_rev.head(15).to_pandas(),
    x='country', y='revenue',
    title='Top 15 Countries by Revenue',
    labels={'revenue': 'Revenue (£)', 'country': 'Country'},
    color='revenue', color_continuous_scale='Blues',
)
fig.show()

uk_rev = country_rev.filter(pl.col('country') == 'United Kingdom')['revenue'][0]
total_rev = country_rev['revenue'].sum()
print(f'UK revenue share: {100*uk_rev/total_rev:.1f}%')

## 6. Product Category Analysis

In [ ]:
with DuckDBAggregator() as duck:
    duck.register_polars('transactions', cleaned)
    cat_rev = duck.agg_product_categories('transactions')

fig = px.bar(
    cat_rev.to_pandas(),
    x='category', y='revenue',
    title='Revenue by Product Category',
    color='revenue', color_continuous_scale='Teal',
)
fig.show()

## 7. Temporal Patterns

In [ ]:
# Day-of-week patterns
with DuckDBAggregator() as duck:
    duck.register_polars('transactions', cleaned)
    dow = duck.query("""
        SELECT
            DAYNAME(invoice_date)       AS day_name,
            DAYOFWEEK(invoice_date)     AS day_num,
            COUNT(DISTINCT invoice_no)  AS orders,
            SUM(quantity * unit_price)  AS revenue
        FROM transactions
        GROUP BY 1, 2
        ORDER BY 2
    """)

fig = px.bar(
    dow.to_pandas(), x='day_name', y='orders',
    title='Orders by Day of Week',
)
fig.show()

In [ ]:
# Hour of day heatmap
with DuckDBAggregator() as duck:
    duck.register_polars('transactions', cleaned)
    hourly = duck.query("""
        SELECT
            DAYOFWEEK(invoice_date)     AS day_num,
            DAYNAME(invoice_date)       AS day_name,
            HOUR(invoice_date)          AS hour,
            COUNT(DISTINCT invoice_no)  AS orders
        FROM transactions
        GROUP BY 1, 2, 3
        ORDER BY 1, 3
    """)

pivot = hourly.pivot(
    values='orders',
    index='day_name',
    on='hour',
    aggregate_function='sum'
)

import plotly.graph_objects as go
z = pivot.drop('day_name').to_numpy()
days = pivot['day_name'].to_list()
hours = [str(h) for h in range(24)]

fig = go.Figure(go.Heatmap(z=z, x=hours, y=days, colorscale='Blues'))
fig.update_layout(title='Orders Heatmap: Day × Hour', xaxis_title='Hour', yaxis_title='Day')
fig.show()

## 8. Customer Segmentation Preview (RFM Quartiles)

In [ ]:
# Quick RFM quartile segmentation using DuckDB
with DuckDBAggregator() as duck:
    duck.register_polars('transactions', cleaned)
    rfm_base = duck.agg_rfm_base('transactions', observation_end=OBS_END)

print(f'RFM base — {len(rfm_base)} customers')
rfm_base.head(5)

In [ ]:
# BG/NBD input distributions
fig = make_subplots(rows=1, cols=3, subplot_titles=['Frequency', 'Recency (days)', 'Monetary Avg (£)'])
fig.add_trace(go.Histogram(x=rfm_base['frequency'].to_list(), nbinsx=40, name='Frequency'), row=1, col=1)
fig.add_trace(go.Histogram(x=rfm_base['recency_days'].to_list(), nbinsx=40, name='Recency'), row=1, col=2)
fig.add_trace(go.Histogram(
    x=rfm_base.filter(pl.col('monetary_avg') < rfm_base['monetary_avg'].quantile(0.99))['monetary_avg'].to_list(),
    nbinsx=40, name='Monetary'
), row=1, col=3)
fig.update_layout(height=350, title='BG/NBD Model Input Distributions', showlegend=False)
fig.show()

In [ ]:
# Frequency vs Monetary scatter
sample = rfm_base.filter(
    pl.col('monetary_avg') < rfm_base['monetary_avg'].quantile(0.99)
).sample(min(2000, len(rfm_base)))

fig = px.scatter(
    sample.to_pandas(),
    x='frequency', y='monetary_avg',
    opacity=0.5, size_max=6,
    title='Frequency vs Average Order Value',
    labels={'frequency': 'Purchase Frequency', 'monetary_avg': 'Avg Order Value (£)'},
    trendline='ols',
)
fig.show()

In [ ]:
# BG/NBD recency vs frequency plot (standard diagnostic)
fig = px.scatter(
    rfm_base.filter(pl.col('frequency') <= 40).to_pandas(),
    x='recency_days', y='frequency',
    opacity=0.3,
    title='BG/NBD Recency–Frequency Plot',
    labels={'recency_days': 'Recency (days)', 'frequency': 'Frequency'},
)
fig.show()

print('\nKey BG/NBD input statistics:')
print(rfm_base.select(['frequency', 'recency_days', 't_days', 'monetary_avg']).describe())

In [ ]:
# Quick RFM quartile segmentation
rfm_scored = rfm_base.with_columns([
    pl.col('frequency').qcut(4, labels=['1','2','3','4']).alias('F_score'),
    pl.col('monetary_avg').qcut(4, labels=['1','2','3','4']).alias('M_score'),
    # Recency: lower = better → reverse
    pl.col('recency_days').qcut(4, labels=['4','3','2','1']).alias('R_score'),
])

rfm_scored = rfm_scored.with_columns(
    (pl.col('R_score').cast(pl.Int32) +
     pl.col('F_score').cast(pl.Int32) +
     pl.col('M_score').cast(pl.Int32)).alias('RFM_score')
)

score_dist = rfm_scored.group_by('RFM_score').len().sort('RFM_score')
fig = px.bar(score_dist.to_pandas(), x='RFM_score', y='len',
             title='RFM Score Distribution (3–12)',
             labels={'RFM_score': 'RFM Score', 'len': 'Customer Count'})
fig.show()

In [ ]:
print('\n=== EDA Complete ===')
print(f'  Dataset:          UCI Online Retail')
print(f'  Raw rows:         {len(raw_df):,}')
print(f'  Cleaned rows:     {len(cleaned):,}')
print(f'  Unique customers: {cleaned["customer_id"].n_unique():,}')
print(f'  Date range:       {cleaned["invoice_date"].min().date()} – {cleaned["invoice_date"].max().date()}')
print(f'  Total revenue:    £{(cleaned["quantity"] * cleaned["unit_price"]).sum():,.0f}')
print(f'\nNext: notebooks/02_rfm_cohort_analysis.ipynb')